# 股指基差分析

四品种（上证50 / 沪深300 / 中证500 / 中证1000）的期货基差、期权隐含基差，并用**预测分红**修正。
计算逻辑都在 `basis/` 包里，这里只负责调用和看图。

**前置步骤**（命令行，见 README）：

```bash
python -m basis init            # 从 data/ 建库
python -m basis jq-backfill     # 聚宽分红数据（需 JQ_ACCOUNT / JQ_PASS）
python -m basis precompute      # 预测分红落库，没有这步图上没有紫色预测线
```

不需要交互看图时，`python -m basis report` 一条命令就能直接出 HTML + CSV。

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from basis import db, calc, report, dividend
from basis.config import PRODUCTS, SLOTS, SLOT_LABEL

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

START = "2022-07-01"   # ETF期权/SHIBOR 自此才有；只看期货历史可改 "2010-01-01"
END   = None

conn = db.connect()

## 一、数据水位

In [ ]:
db.status(conn)

## 二、四品种批量计算（跑一次，缓存到 `results`）

期权链配对是大头，四品种约 1~2 分钟。后面看图、导出都直接用缓存。

In [ ]:
results = calc.analyze_all(conn, START, END)

In [ ]:
report.cross_product_table(results)   # 当月，中位数口径

## 三、单品种深入

改 `PRODUCT` / `SLOT` 切换，不重新计算。实线=含分红，虚线=除已实现分红，紫色点线=除预测分红（可回测）。

In [ ]:
PRODUCT = "300"   # 50 / 300 / 500 / 1000
SLOT    = "00"    # 00当月 / 01次月 / 02季月1 / 03季月2

res = results[PRODUCT]
name = f"{PRODUCTS[PRODUCT]['label']} {SLOT_LABEL[SLOT]}"
for panel in report.panels_for_slot(SLOT):
    report.figure(res, panel, title=f"{name} · {panel[0]}").show()

In [ ]:
report.summary_stats(res)   # 当月年化受剩余天数放大，只看中位数

## 四、分红预测明细（可选）

最新交易日到当前位次合约到期日之间，模型预测了哪些分红、依据是什么。

In [ ]:
panel = dividend.Panel(conn, PRODUCT)
last = res.dropna(subset=[f"{SLOT}_DTE"]).iloc[-1]
asof = last["date"]
until = asof + pd.Timedelta(days=int(last[f"{SLOT}_DTE"]) - 1)
pts, det = dividend.forecast(panel, asof, until)
print(f"{name}：{asof.date()} → {until.date()} 预测分红 {pts:.2f} 点，共 {len(det)} 笔")
det.sort_values("pts", ascending=False).head(15) if len(det) else det

## 五、导出 HTML / CSV 到 `results/`

In [ ]:
report.render_html(results)
report.export_csv(results)